<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_10_Modular_NLP_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Day 10 — Modular NLP Pipeline

## Objective

Build a reusable NLP pipeline that takes raw text as input,
preprocesses it, converts it into TF-IDF vectors, calculates
cosine similarity, and returns ranked documents.

### Modules
1. PreprocessingModule
2. VectorizerModule
3. Pipeline

In [1]:
import nltk
import string
import re

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
## 1. Preprocessing Module

The PreprocessingModule is responsible only for cleaning and
normalizing text.

It performs:
- Lowercasing
- Tokenization
- Stopword removal
- Punctuation/symbol removal
- Input validation

In [4]:
class PreprocessingModule:
    """
    Preprocesses raw text before vectorization.

    Parameters:
        remove_stopwords (bool):
            Whether to remove English stopwords.

        lowercase (bool):
            Whether to convert text to lowercase.
    """

    def __init__(self, remove_stopwords=True, lowercase=True):
        self.remove_stopwords = remove_stopwords
        self.lowercase = lowercase
        self.stop_words = set(stopwords.words("english"))

    def transform(self, text):
        """
        Transform raw text into cleaned text.

        Parameters:
            text (str): Raw input text.

        Returns:
            str: Cleaned text.
        """

        # Empty input
        if not isinstance(text, str):
            raise TypeError("Input must be a string.")

        if text.strip() == "":
            raise ValueError("Input text cannot be empty.")

        # Single-character input
        if len(text.strip()) == 1:
            raise ValueError("Input must contain more than one character.")

        # Numbers and symbols only
        if not any(char.isalpha() for char in text):
            raise ValueError(
                "Input must contain alphabetic characters."
            )

        # Lowercase
        if self.lowercase:
            text = text.lower()

        # Tokenization
        tokens = word_tokenize(text)

        cleaned_tokens = []

        for word in tokens:

            # Keep alphabetic words only
            if word.isalpha():

                # Remove stopwords
                if self.remove_stopwords and word in self.stop_words:
                    continue

                cleaned_tokens.append(word)

        # Make sure something meaningful remains
        if not cleaned_tokens:
            raise ValueError("Input contains no meaningful words.")

        return " ".join(cleaned_tokens)

In [ ]:
### Testing PreprocessingModule

In [5]:
preprocessor = PreprocessingModule()

test_text = "AI is AMAZING!!! It can solve complex problems in 2026."

processed_text = preprocessor.transform(test_text)

print("Original text:")
print(test_text)

print("\nProcessed text:")
print(processed_text)

Original text:
AI is AMAZING!!! It can solve complex problems in 2026.

Processed text:
ai amazing solve complex problems


In [ ]:
## 2. Vectorizer Module

The VectorizerModule is responsible for:

- Learning the vocabulary from the corpus
- Converting documents into TF-IDF vectors
- Converting queries into vectors
- Calculating cosine similarity

In [6]:
class VectorizerModule:

    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.document_vectors = None
        self.corpus = None

    def fit(self, corpus):
        """
        Fit the TF-IDF vectorizer on the corpus.

        Parameters:
            corpus (list):
                List of preprocessed documents.

        Returns:
            self
        """

        if not corpus:
            raise ValueError("Corpus cannot be empty.")

        self.corpus = corpus

        self.document_vectors = self.vectorizer.fit_transform(corpus)

        return self

    def transform(self, query):
        """
        Convert a processed query into a TF-IDF vector.

        Parameters:
            query (str):
                Preprocessed query.

        Returns:
            Query TF-IDF vector.
        """

        if self.document_vectors is None:
            raise RuntimeError(
                "Vectorizer must be fitted before transform()."
            )

        return self.vectorizer.transform([query])

    def similarity(self, query_vector):
        """
        Calculate cosine similarity between the query
        and all corpus documents.
        """

        if self.document_vectors is None:
            raise RuntimeError(
                "Vectorizer must be fitted before similarity()."
            )

        scores = cosine_similarity(
            query_vector,
            self.document_vectors
        )[0]

        return scores

In [ ]:
### Testing VectorizerModule

In [7]:
test_corpus = [
    "machine learning algorithms",
    "deep learning neural networks",
    "natural language processing",
    "python programming language"
]

In [8]:
vectorizer = VectorizerModule()

vectorizer.fit(test_corpus)

query_vector = vectorizer.transform("machine learning")

scores = vectorizer.similarity(query_vector)

print("Similarity scores:")
print(scores)

Similarity scores:
[0.78648108 0.25649872 0.         0.        ]


In [ ]:
## 3. NLP Pipeline

The Pipeline class connects the preprocessing and vectorization
modules.

Flow:

Raw Query
→ Preprocessing
→ TF-IDF Vectorization
→ Cosine Similarity
→ Ranking
→ Final Results

In [9]:
class Pipeline:

    def __init__(self, preprocessor, vectorizer):
        self.preprocessor = preprocessor
        self.vectorizer = vectorizer

    def run(self, query, corpus):
        """
        Run the complete NLP pipeline.

        Parameters:
            query (str):
                Raw search query.

            corpus (list):
                List of raw documents.

        Returns:
            list:
                Documents ranked by similarity score.
        """

        if not corpus:
            raise ValueError("Corpus cannot be empty.")

        # -----------------------------
        # 1. Preprocess corpus
        # -----------------------------

        processed_corpus = []

        for document in corpus:
            processed_document = self.preprocessor.transform(document)
            processed_corpus.append(processed_document)

        # -----------------------------
        # 2. Preprocess query
        # -----------------------------

        processed_query = self.preprocessor.transform(query)

        # -----------------------------
        # 3. Fit vectorizer
        # -----------------------------

        self.vectorizer.fit(processed_corpus)

        # -----------------------------
        # 4. Convert query to vector
        # -----------------------------

        query_vector = self.vectorizer.transform(processed_query)

        # -----------------------------
        # 5. Calculate similarity
        # -----------------------------

        scores = self.vectorizer.similarity(query_vector)

        # -----------------------------
        # 6. Rank documents
        # -----------------------------

        ranked_results = sorted(
            zip(corpus, scores),
            key=lambda x: x[1],
            reverse=True
        )

        return ranked_results

In [ ]:
## 4. Document Corpus

The pipeline will be tested against a corpus containing 15 documents
covering machine learning, NLP, embeddings, similarity, and AI.

In [10]:
corpus = [
    "Machine learning algorithms learn patterns from training data.",

    "Deep learning uses neural networks with multiple layers.",

    "Natural language processing allows computers to understand human language.",

    "Text preprocessing removes punctuation stopwords and unnecessary noise.",

    "TF IDF converts documents into numerical representations.",

    "Cosine similarity measures similarity between two vectors.",

    "Python is widely used for machine learning and artificial intelligence.",

    "Neural networks are inspired by the structure of the human brain.",

    "Classification algorithms predict categories from input data.",

    "Clustering groups similar data points without labeled examples.",

    "Semantic search finds documents based on meaning rather than exact words.",

    "Word embeddings represent words as numerical vectors.",

    "Data preprocessing improves the quality of machine learning models.",

    "Information retrieval systems rank documents according to relevance.",

    "Artificial intelligence combines algorithms and data to solve complex problems."
]

In [11]:
print("Number of documents:", len(corpus))

Number of documents: 15


In [ ]:
## 5. Creating the Pipeline

In [12]:
preprocessor = PreprocessingModule()

vectorizer = VectorizerModule()

pipeline = Pipeline(
    preprocessor=preprocessor,
    vectorizer=vectorizer
)

In [13]:
query = "machine learning"

results = pipeline.run(query, corpus)

for rank, (document, score) in enumerate(results[:5], start=1):

    print(f"{rank}. Similarity: {score:.4f}")
    print(f"   {document}")
    print()

1. Similarity: 0.4628
   Machine learning algorithms learn patterns from training data.

2. Similarity: 0.4560
   Data preprocessing improves the quality of machine learning models.

3. Similarity: 0.4419
   Python is widely used for machine learning and artificial intelligence.

4. Similarity: 0.1926
   Deep learning uses neural networks with multiple layers.

5. Similarity: 0.0000
   Natural language processing allows computers to understand human language.



In [ ]:
## 6. Testing with Five Queries

The pipeline is tested using five different queries.
For each query, documents are ranked according to cosine similarity.

In [14]:
queries = [
    "machine learning algorithms",
    "natural language processing",
    "deep neural networks",
    "document similarity search",
    "data preprocessing"
]

In [15]:
for query in queries:

    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    results = pipeline.run(query, corpus)

    for rank, (document, score) in enumerate(results[:5], start=1):

        print(
            f"{rank}. Similarity: {score:.4f} | {document}"
        )

    print()

QUERY: machine learning algorithms
1. Similarity: 0.5759 | Machine learning algorithms learn patterns from training data.
2. Similarity: 0.3664 | Data preprocessing improves the quality of machine learning models.
3. Similarity: 0.3551 | Python is widely used for machine learning and artificial intelligence.
4. Similarity: 0.2060 | Classification algorithms predict categories from input data.
5. Similarity: 0.1806 | Artificial intelligence combines algorithms and data to solve complex problems.

QUERY: natural language processing
1. Similarity: 0.7394 | Natural language processing allows computers to understand human language.
2. Similarity: 0.0000 | Machine learning algorithms learn patterns from training data.
3. Similarity: 0.0000 | Deep learning uses neural networks with multiple layers.
4. Similarity: 0.0000 | Text preprocessing removes punctuation stopwords and unnecessary noise.
5. Similarity: 0.0000 | TF IDF converts documents into numerical representations.

QUERY: deep neural

In [ ]:
## 7. Edge Case Testing

The pipeline should gracefully handle invalid queries.

Test cases:
1. Empty string
2. Single-character query
3. Numbers and symbols only

In [16]:
try:
    pipeline.run("", corpus)

except ValueError as e:
    print("Empty string:")
    print(e)

Empty string:
Input text cannot be empty.


In [17]:
try:
    pipeline.run("12345!!!@@@", corpus)

except ValueError as e:
    print("Numbers/symbols only:")
    print(e)

Numbers/symbols only:
Input must contain alphabetic characters.


In [ ]:
## 8. Pipeline Architecture

```text
                RAW TEXT / QUERY
                       |
                       v
            +----------------------+
            | PreprocessingModule  |
            |----------------------|
            | Lowercase            |
            | Tokenization         |
            | Stopword Removal     |
            | Noise Removal        |
            +----------+-----------+
                       |
                       v
                CLEANED TEXT
                       |
                       v
            +----------------------+
            |   VectorizerModule   |
            |----------------------|
            | TF-IDF Vectorization |
            | Cosine Similarity    |
            +----------+-----------+
                       |
                       v
              SIMILARITY SCORES
                       |
                       v
            +----------------------+
            |       Pipeline       |
            |----------------------|
            | Ranking              |
            | Result Formatting    |
            +----------+-----------+
                       |
                       v
                 RANKED RESULTS

In [ ]:
## 10. Conclusion

The modular NLP pipeline successfully connects text preprocessing,
TF-IDF vectorization, cosine similarity, and document ranking into
a reusable system.

The pipeline was tested on:
- 15 documents
- 5 different queries
- 3 invalid-input edge cases

The modular design allows each component to be independently tested,
debugged, replaced, and extended.